# Paired Data Training: Source to Target Transformation

This notebook trains a model to transform audio from a source domain to a target domain using paired data.

**Strategy:**
- Load paired source files (e.g., `audio_mel`) and target files (e.g., `classical`)
- Train with paired data to learn domain transformation
- Monitor loss and checkpoints


In [2]:
import torch
from pathlib import Path
import glob
from training.dataloader import RandomPairMelDataset, PairedMelDataset
from torch.utils.data import DataLoader
from training.training import TrainingConfig, TrainingPipeline

print("="*70)
print("SETUP: Imports Complete")
print("="*70)
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

SETUP: Imports Complete
✓ PyTorch version: 2.9.0+cpu
✓ CUDA available: False


In [3]:
# Setup paths - CONFIGURE THESE
source_path = r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\batch_output\audio_mel"
target_path = r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical"

root = Path(r"c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift")

# Get file lists
source_files = sorted(glob.glob(str(Path(source_path) / "*.pt")))  # Source files
target_files = sorted(glob.glob(str(Path(target_path) / "*.pt")))  # Target files

print("\n" + "="*70)
print("AVAILABLE DATA")
print("="*70)
print(f"Source files available: {len(source_files)}")
print(f"Target files available: {len(target_files)}")
print(f"\nFirst 3 source files:")
for f in source_files[:3]:
    print(f"  - {Path(f).name}")
print(f"\nFirst 3 target files:")
for f in target_files[:3]:
    print(f"  - {Path(f).name}")


AVAILABLE DATA
Source files available: 413
Target files available: 413

First 3 source files:
  - 00000_Preludes Book 2 -_synth_mel.pt
  - 00001_Preludes Book 2 - La puerta del Vino_synth_mel.pt
  - 00002_Sonata No 1 in F Minor Op 2 No 1 - I Allegro_synth_mel.pt

First 3 target files:
  - 00000_instrum_mel.pt
  - 00001_instrum_mel.pt
  - 00002_instrum_mel.pt


In [4]:
# PAIR SOURCE AND TARGET FILES - LIMITED FOR QUICK TESTING
# Set this to limit training data (e.g., 2-5 for quick tests, None for all)
max_targets = 20  # Change this value for quick testing

# For paired data: ensure source_files and target_files have corresponding pairs
# They should be aligned by index or name

# Option 1: If files are already ordered and paired by index
if len(source_files) == len(target_files):
    print("\n✓ Files are equally sized - using direct pairing by index")
    paired_source_files = source_files
    paired_target_files = target_files
else:
    # Option 2: Try to match by name pattern
    print(f"\n⚠ File count mismatch: {len(source_files)} sources vs {len(target_files)} targets")
    print("Attempting to pair by matching indices...")
    min_len = min(len(source_files), len(target_files))
    paired_source_files = source_files[:min_len]
    paired_target_files = target_files[:min_len]
    print(f"Using {min_len} pairs")

# Apply limit for quick testing
if max_targets is not None:
    paired_source_files = paired_source_files[:max_targets]
    paired_target_files = paired_target_files[:max_targets]

print("\n" + "="*70)
print("PAIRED DATA")
print("="*70)
print(f"Total pairs: {len(paired_source_files)}")
if max_targets:
    print(f"⚡ Limited to {max_targets} targets for quick testing")
print(f"Source → Target pairs (first 5):")
for i, (src, tgt) in enumerate(zip(paired_source_files[:5], paired_target_files[:5])):
    print(f"  {i+1}. {Path(src).name} → {Path(tgt).name}")


✓ Files are equally sized - using direct pairing by index

PAIRED DATA
Total pairs: 20
⚡ Limited to 20 targets for quick testing
Source → Target pairs (first 5):
  1. 00000_Preludes Book 2 -_synth_mel.pt → 00000_instrum_mel.pt
  2. 00001_Preludes Book 2 - La puerta del Vino_synth_mel.pt → 00001_instrum_mel.pt
  3. 00002_Sonata No 1 in F Minor Op 2 No 1 - I Allegro_synth_mel.pt → 00002_instrum_mel.pt
  4. 00003_Sonata No 1 in F Minor Op 2 No 1 - II Adagio_synth_mel.pt → 00003_instrum_mel.pt
  5. 00004_Sonata No 1 in F Minor Op 2 No 1 - III Menuetto Al_synth_mel.pt → 00004_instrum_mel.pt


In [5]:
# Load sample to check shapes
sample_source = torch.load(paired_source_files[0])
sample_target = torch.load(paired_target_files[0])

print("\n" + "="*70)
print("DATA SAMPLE CHECK")
print("="*70)

# Handle dict vs tensor
if isinstance(sample_source, dict):
    for k in ('mel', 'spec', 'melspec', 'x'):
        if k in sample_source:
            sample_source = sample_source[k]
            break

if isinstance(sample_target, dict):
    for k in ('mel', 'spec', 'melspec', 'x'):
        if k in sample_target:
            sample_target = sample_target[k]
            break

print(f"Source shape: {sample_source.shape}")
print(f"Target shape: {sample_target.shape}")
print(f"Source dtype: {sample_source.dtype}")
print(f"Source min/max: [{sample_source.min():.3f}, {sample_source.max():.3f}]")
print(f"Target min/max: [{sample_target.min():.3f}, {sample_target.max():.3f}]")
print(f"\n✓ Paired data ready for training")


DATA SAMPLE CHECK
Source shape: torch.Size([1, 100, 3003])
Target shape: torch.Size([1, 100, 2813])
Source dtype: torch.float32
Source min/max: [0.000, 51996.043]
Target min/max: [0.000, 4070.261]

✓ Paired data ready for training


In [6]:
# TRAINING CONFIG FOR PAIRED DATA
config = TrainingConfig(
    num_epochs=20,           # Reduced epochs
    batch_size=2,            # Batch size 1 to maximize steps
    learning_rate=5e-4,      # Higher LR for faster convergence
    weight_decay=1e-5,       # Reduce regularization
    grad_clip_norm=1.0,      # Prevent explosion
    checkpoint_interval=1,   # Save every epoch
    checkpoint_dir="checkpoints/paired_training",
    device="cuda" if torch.cuda.is_available() else "cpu",
    max_time=1024,           # Max mel time steps
    patch_height=10,
    patch_width=16,
    embed_dim=128,           # Smaller model
    num_blocks=2,            # Fewer blocks
    num_heads=2,             # Fewer heads
    hidden_dim=512,
    dropout=0.1,
    num_genres=3,            # Rock, Classical, Synth
    in_channels=1,
)

print("\n" + "="*70)
print("TRAINING CONFIG")
print("="*70)
print(f"Epochs: {config.num_epochs}")
print(f"Batch size: {config.batch_size}")
print(f"Learning rate: {config.learning_rate}")
print(f"Device: {config.device}")
print(f"Model size: embed_dim={config.embed_dim}, blocks={config.num_blocks}, heads={config.num_heads}")
print(f"Checkpoint dir: {config.checkpoint_dir}")
print(f"Num genres: {config.num_genres} (Rock, Classical, Synth)")


TRAINING CONFIG
Epochs: 20
Batch size: 2
Learning rate: 0.0005
Device: cpu
Model size: embed_dim=128, blocks=2, heads=2
Checkpoint dir: checkpoints/paired_training
Num genres: 3 (Rock, Classical, Synth)


In [7]:
# Initialize pipeline
pipeline = TrainingPipeline(config)

print("\n" + "="*70)
print("PIPELINE INITIALIZED")
print("="*70)
print(f"✓ Model loaded to {config.device}")
print(f"✓ Optimizer: AdamW")
print(f"✓ Scheduler: ReduceLROnPlateau")

# Count parameters
total_params = sum(p.numel() for p in pipeline.flow.parameters())
trainable_params = sum(p.numel() for p in pipeline.flow.parameters() if p.requires_grad)
print(f"\nModel parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")


PIPELINE INITIALIZED
✓ Model loaded to cpu
✓ Optimizer: AdamW
✓ Scheduler: ReduceLROnPlateau

Model parameters:
  Total: 1,488,928
  Trainable: 1,488,928


In [8]:
# Import collate function (reload module if needed)
import importlib
import training.dataloader
importlib.reload(training.dataloader)
from training.dataloader import collate_variable_length_mel

# Setup dataloader
# Use PairedMelDataset for deterministic paired overfitting (same pairs each epoch)
# Use RandomPairMelDataset for random pairing (different target each time)

print("\n" + "="*70)
print("CHOOSING DATASET TYPE")
print("="*70)

# Set to True to test overfitting on paired data (deterministic)
# Set to False for random pairing (standard training)
use_paired_dataset = True
repeat_dataset = 2  # Repeat dataset N times to force overfitting on small data

if use_paired_dataset:
    print("✓ Using PairedMelDataset (DETERMINISTIC PAIRING)")
    print(f"  → Same source-target pairs each epoch")
    print(f"  → Repeat factor: {repeat_dataset}x (for overfitting on small datasets)")
    dataset = PairedMelDataset(paired_source_files, paired_target_files, repeat=repeat_dataset)
else:
    print("✓ Using RandomPairMelDataset (RANDOM PAIRING)")
    print(f"  → Random target for each source")
    dataset = RandomPairMelDataset(paired_source_files, paired_target_files)

# Create loader with custom collate function (handles variable-length mels)
loader = DataLoader(
    dataset, 
    batch_size=config.batch_size, 
    shuffle=True,
    collate_fn=collate_variable_length_mel  # Custom collate for padding
)

print("\n" + "="*70)
print("DATALOADER READY")
print("="*70)
print(f"Dataset type: {'PairedMelDataset' if use_paired_dataset else 'RandomPairMelDataset'}")
print(f"Total batches per epoch: {len(loader)}")
print(f"Batch size: {config.batch_size}")
print(f"Total pairs: {len(paired_source_files)}")
if use_paired_dataset:
    print(f"Repeat factor: {repeat_dataset}x")
    print(f"Effective dataset size: {len(dataset)}")
print(f"\n✓ Custom collate function: collate_variable_length_mel (handles padding)")

# Test batch
batch = next(iter(loader))
x0_batch, x1_batch, mask_batch = batch
print(f"\nBatch shapes (with padding):")
print(f"  x0 (source): {x0_batch.shape}")
print(f"  x1 (target): {x1_batch.shape}")
print(f"  mask: {mask_batch.shape}")
print(f"  mask dtype: {mask_batch.dtype}")
print(f"\n✓ Data ready for training!")


CHOOSING DATASET TYPE
✓ Using PairedMelDataset (DETERMINISTIC PAIRING)
  → Same source-target pairs each epoch
  → Repeat factor: 2x (for overfitting on small datasets)

DATALOADER READY
Dataset type: PairedMelDataset
Total batches per epoch: 20
Batch size: 2
Total pairs: 20
Repeat factor: 2x
Effective dataset size: 40

✓ Custom collate function: collate_variable_length_mel (handles padding)

Batch shapes (with padding):
  x0 (source): torch.Size([2, 1, 100, 3694])
  x1 (target): torch.Size([2, 1, 100, 3694])
  mask: torch.Size([2, 3694])
  mask dtype: torch.bool

✓ Data ready for training!


## Training Loop

Now we'll train the model. This will learn to transform source audio to match the target domain using paired data.


In [11]:
# START TRAINING
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Source path: {source_path}")
print(f"Target path: {target_path}")
print(f"Training pairs: {len(paired_source_files)}")
print(f"Dataset type: {'PairedMelDataset (OVERFITTING MODE)' if use_paired_dataset else 'RandomPairMelDataset'}")
if use_paired_dataset:
    print(f"Repeat factor: {repeat_dataset}x")
    print(f"Effective size: {len(dataset)}")
print(f"Total epochs: {config.num_epochs}")
print(f"Expected batches: {len(loader) * config.num_epochs}")
print("="*70 + "\n")

try:
    pipeline.train(loader)
    print("\n" + "="*70)
    print("✓ TRAINING COMPLETED SUCCESSFULLY")
    print("="*70)
except KeyboardInterrupt:
    print("\n" + "="*70)
    print("⚠ Training interrupted by user")
    print("="*70)
except Exception as e:
    print(f"\n❌ Error during training: {e}")
    import traceback
    traceback.print_exc()


STARTING TRAINING
Source path: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\batch_output\audio_mel
Target path: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical
Training pairs: 20
Dataset type: PairedMelDataset (OVERFITTING MODE)
Repeat factor: 2x
Effective size: 40
Total epochs: 20
Expected batches: 400


Epoch 1/20 started
Starting batch shapes:
  x0: torch.Size([2, 1, 100, 3141])
  x1: torch.Size([2, 1, 100, 3141])
  mask: torch.Size([2, 3141])

❌ Error during training: The size of tensor a (100) must match the size of tensor b (2) at non-singleton dimension 2


Traceback (most recent call last):
  File "C:\Users\Dhanuja\AppData\Local\Temp\ipykernel_30800\309474249.py", line 17, in <module>
    pipeline.train(loader)
  File "C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\training\training.py", line 172, in train
    x0 = x0 * mask
RuntimeError: The size of tensor a (100) must match the size of tensor b (2) at non-singleton dimension 2


In [10]:
# Check checkpoints
import os

checkpoint_dir = Path(config.checkpoint_dir)
if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob("*.pt"))
    print("\n" + "="*70)
    print("SAVED CHECKPOINTS")
    print("="*70)
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / (1024**2)
        print(f"  {ckpt.name} ({size_mb:.1f} MB)")
    print(f"\nTotal checkpoints: {len(checkpoints)}")
    print(f"Latest: {checkpoints[-1].name if checkpoints else 'None'}")
else:
    print(f"\n⚠ Checkpoint directory not found: {checkpoint_dir}")


SAVED CHECKPOINTS
  best_model.pt (17.1 MB)
  checkpoint_epoch_001.pt (17.1 MB)
  checkpoint_epoch_002.pt (17.1 MB)
  checkpoint_epoch_003.pt (17.1 MB)
  checkpoint_epoch_004.pt (17.1 MB)
  checkpoint_epoch_005.pt (17.1 MB)
  checkpoint_epoch_006.pt (17.1 MB)
  checkpoint_epoch_007.pt (17.1 MB)
  checkpoint_epoch_008.pt (17.1 MB)
  checkpoint_epoch_009.pt (17.1 MB)
  checkpoint_epoch_010.pt (17.1 MB)
  checkpoint_epoch_011.pt (17.1 MB)
  checkpoint_epoch_012.pt (17.1 MB)
  checkpoint_epoch_013.pt (17.1 MB)
  checkpoint_epoch_014.pt (17.1 MB)
  checkpoint_epoch_015.pt (17.1 MB)
  checkpoint_epoch_016.pt (17.1 MB)
  checkpoint_epoch_017.pt (17.1 MB)
  checkpoint_epoch_018.pt (17.1 MB)
  checkpoint_epoch_019.pt (17.1 MB)
  checkpoint_epoch_020.pt (17.1 MB)

Total checkpoints: 21
Latest: checkpoint_epoch_020.pt


## Next Steps

After training:
1. Verify paired data alignment by checking source-target file correspondence
2. Use the saved checkpoint to perform inference on your source files
3. Listen to the output to verify target domain transformation
4. Adjust hyperparameters (learning rate, epochs) based on results
5. Try different source/target path combinations
